# 05 - 调试技巧与常见问题

本 notebook 汇总 MiniMind-O 训练/推理/评估中的常见错误、排查方法和调试技巧。

## 1. 训练前检查清单

在启动任何训练前，先确认：

- [ ] `python -c "import torch; print(torch.cuda.is_available())"` 返回 True
- [ ] GPU 显存足够（113M dense A2A 约 24-30GB，T2A 约 16-20GB）
- [ ] `model/SenseVoiceSmall`、`model/siglip2-base-p32-256-ve`、`model/mimi` 已下载
- [ ] 数据 parquet 存在，且 schema 包含 `conversations` 列
- [ ] `out/llm_768.pth` 或对应基座权重存在（除非 from_weight=none）
- [ ] `checkpoints/` 和 `out/` 目录可写
- [ ] 先跑 DRY_RUN=1 检查命令是否正确

## 2. 常见错误与排查

### 2.1 `RuntimeError: probability tensor contains either inf, nan or element < 0`

**原因**：推理时 logits 在 fp16 下溢出，softmax 后概率非有限。

**解决**：
- 使用 `--dtype bf16` 或 `--dtype fp32`。
- 确认 `model_omni.py` 中 `stream_generate` 有 `logits.float()`（409、445 行）。

### 2.2 `RuntimeError: NCCL operation failed` / `NCCL watchdog timeout`

**原因**：DDP buffer broadcast 在 Transformer 训练中产生大量同步。

**解决**：
- 设置 `DDP_BROADCAST_BUFFERS=0`（v3/v4 默认已设置）。
- 检查各 rank 的 batch 数量是否一致（`validate_equal_batch_count`）。

### 2.3 resume 后 loss 突然涨到 10+

**原因**：torch.compile 包装的模型直接 load_state_dict 会静默跳过大量真实权重。

**解决**：
- 确认 resume 代码使用 `raw_model = getattr(model, '_orig_mod', model)` 加载。
- 检查 `checkpoints/*_resume.pth` 中模型权重的 dtype 是否为 fp32。

### 2.4 OOM（显存不足）

**排查**：
- 降低 `batch_size`（A2A 阶段显存压力最大）。
- 开启 `gradient_checkpointing=1`。
- 关闭 `use_compile=0`（compile 会增加显存碎片）。
- 缩短 `max_seq_len`。

### 2.5 模型加载 missing/unexpected keys

**原因**：模型架构参数与权重不匹配。

**解决**：
- 检查 `hidden_size`, `num_hidden_layers`, `use_moe` 等参数。
- 注意 `OmniConfig` 的默认值可能与权重训练时不同。

### 2.6 生成音频为空或全是噪声

**排查**：
- 检查 `audio_frames` 是否为空。
- 检查 Mimi decode 前是否过滤了 >=2049 的 code。
- 检查 `special_code_rate` 是否过高。
- 尝试降低 `temperature` 和 `top_k`。

## 3. 显存分析技巧

使用 PyTorch 自带工具查看显存占用。

In [ ]:
import torch

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    print(f"当前显存: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"预留显存: {torch.cuda.memory_reserved()/1024**3:.2f} GB")
    print(f"峰值显存: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
else:
    print("无 GPU")

## 4. 训练曲线分析

健康的训练曲线特征：
- text loss 在前几百 step 明显下降。
- audio loss 缓慢下降，不震荡。
- val loss 不突然反弹。
- 没有 `FloatingPointError: Non-finite loss`。

异常信号：
- `rvq_layer_0_loss` 远大于其他层 → 可能 RVQ 权重分配不合理。
- `audio_loss` 不下降但 `text_loss` 下降 → A2A 数据或 Talker 初始化问题。
- Stage6 val loss 高于 Stage3 → 图像阶段遗忘 audio（见 Run C 诊断）。

## 5. 调试代码片段

### 5.1 打印模型某一层的梯度范数

In [ ]:
# 为模型注册 hook，打印每层梯度范数
def register_grad_hooks(model):
    handles = []
    for name, param in model.named_parameters():
        if param.requires_grad:
            def hook(grad, name=name):
                print(f"{name}: grad norm {grad.norm().item():.6f}")
            handles.append(param.register_hook(hook))
    return handles

print("register_grad_hooks 已定义，可在训练循环中调用。")

### 5.2 检查输入数据是否有非法值

In [ ]:
def sanity_check_batch(input_ids, labels, audio_labels):
    print(f"input_ids range: [{input_ids.min()}, {input_ids.max()}]")
    print(f"labels non -100 count: {(labels != -100).sum().item()}")
    print(f"audio_labels non -100 count: {(audio_labels != -100).sum().item()}")
    # 检查 audio_labels 是否包含超出 vocab 的值
    valid = audio_labels[audio_labels != -100]
    if valid.numel() > 0:
        print(f"audio_labels valid range: [{valid.min()}, {valid.max()}]")

print("sanity_check_batch 已定义。")

## 6. 进阶：复现 Run C 的诊断路径

如果 audio CER 下降，可按以下步骤定位：

1. 评估 Stage3 `_a2a_full` 检查点：`out/sft_full_muon_v3_a2a_full_768.pth`。
2. 比较 Stage3 与 Stage1 T2A 的 audio CER，确认问题出现阶段。
3. 若 Stage3 已差，尝试 uniform RVQ weights（Run E probe）。
4. 若 uniform 无效，尝试 `loss_norm=local`。
5. 仍无效则检查 A2A 数据质量 / Mimi decode 设置。

## 7. 推荐阅读

- `docs/training_summaries/full_train_comparison_abcd_v5probe_20260613.md`：最新 A/B/C/D 对比
- `docs/training_plans/next_model_training_runE_20260613.md`：Run E RVQ uniform 探针计划
- `TRAINING_TUTORIAL.md`：官方训练教程
- `README.md`：项目总览